# Baby Products P4 strict temporal graph — G2-C/G2-D

Notebook self-contained này chuyển checksummed `Baby_Products` 0-core artifact thành training-only P4 graph cùng warm-start validation/test artifact. Notebook xử lý deterministic bằng SQLite, chỉ suy mapping và graph statistic từ training positive, ghi mọi OOV exclusion và chạy bounded chunked full-catalog traversal mà không train model.

Các cutoff `t1`/`t2` được execute như candidate G2-C hiện tại. Run thành công tạo evidence để review; nó không âm thầm freeze cutoff hoặc tạo recommendation/scalability result.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_PATH = DRIVE_ROOT / 'Phase2_Amazon_Audit' / 'raw' / 'Baby_Products.csv.gz'
OUTPUT_DIR = DRIVE_ROOT / 'Phase2_Amazon_Audit' / 'g2c_baby_p4'
DATABASE_PATH = Path('/content/baby_p4_g2c.sqlite')

T1_MS = 1628643414042
T2_MS = 1658002729837
MIN_TRAIN_USER_DEGREE = 1
MIN_TRAIN_ITEM_DEGREE = 1
DRY_RUN_TARGETS = 100
EVALUATION_CHUNK_SIZE = 16384
FORCE_REBUILD_G2C = False
NOTEBOOK_REVISION = 'g2c-baby-p4-self-contained-v1-2026-09-02'
IMPLEMENTATION_SHA256 = '18dd46ad142491891ce6d5b1c4add4a8e6e5953a22c46bd429b3732798117998'

assert RAW_PATH.is_file(), f'Missing raw artifact: {RAW_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raw:', RAW_PATH)
print('Output:', OUTPUT_DIR)


## Toàn bộ implementation



In [ ]:
import csv
import datetime as dt
import gzip
import hashlib
import io
import json
import os
import resource
import sqlite3
import time
from array import array
from pathlib import Path


EXPECTED_COLUMNS = ("user_id", "parent_asin", "rating", "timestamp")


def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def open_text(path):
    path = Path(path)
    if path.name.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", newline="")
    return path.open("rt", encoding="utf-8", newline="")


def scalar(connection, query, parameters=()):
    value = connection.execute(query, parameters).fetchone()[0]
    return int(value or 0)


def fraction(numerator, denominator):
    return numerator / denominator if denominator else None


def histogram_summary(rows):
    histogram = [(int(value), int(count)) for value, count in rows]
    count = sum(n for _, n in histogram)
    total = sum(value * n for value, n in histogram)
    if not count:
        return {"count": 0, "min": 0, "max": 0, "mean": 0.0, "p50": 0, "p90": 0, "p95": 0, "p99": 0, "singletons": 0, "singleton_rate": 0.0}

    def percentile(probability):
        target = probability * (count - 1)
        lower_rank = int(target)
        upper_rank = lower_rank if target == lower_rank else lower_rank + 1
        found = []
        cumulative = 0
        for value, n in histogram:
            next_cumulative = cumulative + n
            for rank in (lower_rank, upper_rank):
                if len(found) < 2 and cumulative <= rank < next_cumulative:
                    found.append(value)
            cumulative = next_cumulative
            if len(found) == 2:
                break
        if len(found) == 1:
            found.append(found[0])
        weight = target - lower_rank
        return found[0] + weight * (found[1] - found[0])

    singletons = next((n for value, n in histogram if value == 1), 0)
    return {
        "count": count,
        "min": histogram[0][0],
        "max": histogram[-1][0],
        "mean": total / count,
        "p50": percentile(0.50),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "p99": percentile(0.99),
        "singletons": singletons,
        "singleton_rate": singletons / count,
    }


def export_query_gzip(connection, query, output_path, header):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    row_count = 0
    with output_path.open("wb") as raw_handle:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw_handle, mtime=0) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8", newline="") as handle:
                writer = csv.writer(handle)
                writer.writerow(header)
                for row in connection.execute(query):
                    writer.writerow(row)
                    row_count += 1
    return {
        "path": str(output_path),
        "rows": row_count,
        "bytes": output_path.stat().st_size,
        "sha256": sha256_file(output_path),
    }


def connected_component_summary(connection, user_count, item_count):
    node_count = user_count + item_count
    parent = array("I", range(node_count))
    size = array("I", [1]) * node_count

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root == right_root:
            return
        if size[left_root] < size[right_root]:
            left_root, right_root = right_root, left_root
        parent[right_root] = left_root
        size[left_root] += size[right_root]

    for user_idx, item_idx in connection.execute("SELECT user_idx, item_idx FROM mapped_train"):
        union(int(user_idx), user_count + int(item_idx))

    counts = {}
    for node in range(node_count):
        root = find(node)
        counts[root] = counts.get(root, 0) + 1
    values = sorted(counts.values(), reverse=True)
    return {
        "components": len(values),
        "largest_component_nodes": values[0] if values else 0,
        "largest_component_fraction": fraction(values[0], node_count) if values else None,
        "singleton_components": sum(value == 1 for value in values),
        "node_count": node_count,
    }


def partition_ledger(connection, where_clause, parameters):
    base = f"FROM p4_events e LEFT JOIN user_map u ON u.user_id=e.user_id LEFT JOIN item_map i ON i.item_id=e.item_id WHERE {where_clause}"
    candidate_rows = scalar(connection, "SELECT COUNT(*) " + base, parameters)
    warm_rows = scalar(connection, "SELECT COUNT(*) " + base + " AND u.user_idx IS NOT NULL AND i.item_idx IS NOT NULL", parameters)
    unseen_user_only = scalar(connection, "SELECT COUNT(*) " + base + " AND u.user_idx IS NULL AND i.item_idx IS NOT NULL", parameters)
    unseen_item_only = scalar(connection, "SELECT COUNT(*) " + base + " AND u.user_idx IS NOT NULL AND i.item_idx IS NULL", parameters)
    unseen_both = scalar(connection, "SELECT COUNT(*) " + base + " AND u.user_idx IS NULL AND i.item_idx IS NULL", parameters)
    return {
        "candidate_rows": candidate_rows,
        "warm_rows": warm_rows,
        "warm_retention": fraction(warm_rows, candidate_rows),
        "excluded_unseen_user_only": unseen_user_only,
        "excluded_unseen_item_only": unseen_item_only,
        "excluded_unseen_user_and_item": unseen_both,
        "excluded_total": candidate_rows - warm_rows,
        "reconciles": warm_rows + unseen_user_only + unseen_item_only + unseen_both == candidate_rows,
    }


def run_dry_feasibility(connection, item_count, target_limit, chunk_size):
    targets = list(connection.execute(
        "SELECT user_idx, item_idx, timestamp_ms, candidate_count FROM warm_validation ORDER BY source_row LIMIT ?",
        (target_limit,),
    ))
    started = time.perf_counter()
    eligible_total = 0
    comparisons = 0
    for user_idx, target_item_idx, timestamp_ms, expected_count in targets:
        prior = {
            int(row[0])
            for row in connection.execute(
                "SELECT item_idx FROM mapped_p4 WHERE user_idx=? AND timestamp_ms<?",
                (user_idx, timestamp_ms),
            )
        }
        if int(target_item_idx) in prior:
            raise AssertionError("Target item was incorrectly present in prior history")
        actual_count = 0
        for start in range(0, item_count, chunk_size):
            stop = min(start + chunk_size, item_count)
            actual_count += sum(item_idx not in prior for item_idx in range(start, stop))
            comparisons += stop - start
        if actual_count != int(expected_count):
            raise AssertionError(f"Candidate count mismatch: {actual_count} != {expected_count}")
        eligible_total += actual_count
    elapsed = time.perf_counter() - started
    rss_raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    rss_mb = rss_raw / 1024.0 if os.name != "darwin" else rss_raw / (1024.0 * 1024.0)
    return {
        "status": "BOUNDED_EVALUATOR_TRAVERSAL_EXECUTED; NO MODEL OR QUALITY METRIC",
        "targets_requested": target_limit,
        "targets_executed": len(targets),
        "catalog_items": item_count,
        "chunk_size": chunk_size,
        "catalog_comparisons": comparisons,
        "eligible_candidates_counted": eligible_total,
        "wall_seconds": elapsed,
        "process_peak_rss_mb": rss_mb,
        "invariants": {
            "target_not_in_prior_history": True,
            "chunked_count_matches_manifest_count": True,
        },
        "claim_boundary": "Pipeline/evaluator traversal feasibility only; not comparable profiling, recommendation quality, or scalability evidence.",
    }


def build_g2c_artifacts(input_path, output_dir, database_path, t1_ms, t2_ms, min_user_degree=1, min_item_degree=1, dry_run_targets=100, evaluation_chunk_size=16384, cleanup_database=True):
    input_path, output_dir, database_path = Path(input_path), Path(output_dir), Path(database_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    if database_path.exists():
        database_path.unlink()

    connection = sqlite3.connect(database_path)
    connection.execute("PRAGMA journal_mode=OFF")
    connection.execute("PRAGMA synchronous=OFF")
    connection.execute("PRAGMA temp_store=FILE")
    connection.execute("CREATE TABLE raw_p4(user_id TEXT NOT NULL, item_id TEXT NOT NULL, rating REAL NOT NULL, timestamp_ms INTEGER NOT NULL, source_row INTEGER NOT NULL)")

    raw_rows = invalid_rows = anomaly_zero_rows = p4_rows = 0
    batch = []
    with open_text(input_path) as handle:
        reader = csv.DictReader(handle)
        if tuple(reader.fieldnames or ()) != EXPECTED_COLUMNS:
            raise ValueError(f"Expected {EXPECTED_COLUMNS}, found {reader.fieldnames}")
        for source_row, row in enumerate(reader, start=2):
            raw_rows += 1
            user_id, item_id = (row.get("user_id") or "").strip(), (row.get("parent_asin") or "").strip()
            try:
                rating, timestamp_ms = float(row.get("rating", "")), int(row.get("timestamp", ""))
            except (TypeError, ValueError):
                invalid_rows += 1
                continue
            if not user_id or not item_id:
                invalid_rows += 1
                continue
            if rating == 0.0:
                anomaly_zero_rows += 1
            if 4.0 <= rating <= 5.0:
                batch.append((user_id, item_id, rating, timestamp_ms, source_row))
                p4_rows += 1
                if len(batch) >= 50000:
                    connection.executemany("INSERT INTO raw_p4 VALUES(?,?,?,?,?)", batch)
                    connection.commit()
                    batch.clear()
    if batch:
        connection.executemany("INSERT INTO raw_p4 VALUES(?,?,?,?,?)", batch)
        connection.commit()

    connection.execute("CREATE INDEX raw_pair_idx ON raw_p4(user_id,item_id)")
    connection.execute("CREATE TABLE p4_events AS SELECT user_id,item_id,rating,timestamp_ms,source_row FROM (SELECT *, ROW_NUMBER() OVER(PARTITION BY user_id,item_id ORDER BY timestamp_ms,source_row) AS pair_rank FROM raw_p4) WHERE pair_rank=1")
    connection.execute("CREATE INDEX p4_time_idx ON p4_events(timestamp_ms)")
    connection.execute("CREATE INDEX p4_user_idx ON p4_events(user_id)")
    connection.execute("CREATE INDEX p4_item_idx ON p4_events(item_id)")
    p4_deduplicated = scalar(connection, "SELECT COUNT(*) FROM p4_events")

    connection.execute("CREATE TABLE train_edges AS SELECT * FROM p4_events WHERE timestamp_ms < ?", (t1_ms,))
    filter_ledger = []
    iteration = 0
    while True:
        before = scalar(connection, "SELECT COUNT(*) FROM train_edges")
        if min_user_degree <= 1 and min_item_degree <= 1:
            removed = 0
        else:
            connection.execute(
                "DELETE FROM train_edges WHERE user_id IN (SELECT user_id FROM train_edges GROUP BY user_id HAVING COUNT(*) < ?) OR item_id IN (SELECT item_id FROM train_edges GROUP BY item_id HAVING COUNT(*) < ?)",
                (min_user_degree, min_item_degree),
            )
            connection.commit()
            removed = before - scalar(connection, "SELECT COUNT(*) FROM train_edges")
        filter_ledger.append({"iteration": iteration, "before_edges": before, "removed_edges": removed, "after_edges": before - removed})
        if removed == 0:
            break
        iteration += 1

    connection.execute("CREATE INDEX train_user_idx ON train_edges(user_id)")
    connection.execute("CREATE INDEX train_item_idx ON train_edges(item_id)")
    connection.execute("CREATE TABLE user_map AS SELECT user_id, ROW_NUMBER() OVER(ORDER BY user_id)-1 AS user_idx FROM (SELECT DISTINCT user_id FROM train_edges)")
    connection.execute("CREATE TABLE item_map AS SELECT item_id, ROW_NUMBER() OVER(ORDER BY item_id)-1 AS item_idx FROM (SELECT DISTINCT item_id FROM train_edges)")
    connection.execute("CREATE UNIQUE INDEX user_map_id ON user_map(user_id)")
    connection.execute("CREATE UNIQUE INDEX item_map_id ON item_map(item_id)")
    connection.execute("CREATE TABLE mapped_train AS SELECT u.user_idx,i.item_idx,e.rating,e.timestamp_ms,e.source_row FROM train_edges e JOIN user_map u USING(user_id) JOIN item_map i USING(item_id)")
    connection.execute("CREATE INDEX mapped_train_user ON mapped_train(user_idx)")
    user_count, item_count = scalar(connection, "SELECT COUNT(*) FROM user_map"), scalar(connection, "SELECT COUNT(*) FROM item_map")
    train_count = scalar(connection, "SELECT COUNT(*) FROM mapped_train")

    connection.execute("CREATE TABLE mapped_p4 AS SELECT u.user_idx,i.item_idx,e.rating,e.timestamp_ms,e.source_row FROM p4_events e JOIN user_map u USING(user_id) JOIN item_map i USING(item_id)")
    connection.execute("CREATE INDEX mapped_p4_user_time ON mapped_p4(user_idx,timestamp_ms)")
    connection.execute("CREATE TABLE user_time_counts AS SELECT user_idx,timestamp_ms,COUNT(*) AS event_count FROM mapped_p4 GROUP BY user_idx,timestamp_ms")
    connection.execute("CREATE TABLE user_time_prefix AS SELECT user_idx,timestamp_ms,COALESCE(SUM(event_count) OVER(PARTITION BY user_idx ORDER BY timestamp_ms ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),0) AS prior_count FROM user_time_counts")
    connection.execute("CREATE UNIQUE INDEX prefix_idx ON user_time_prefix(user_idx,timestamp_ms)")

    for table_name, where_clause, parameters in (
        ("warm_validation", "m.timestamp_ms>=? AND m.timestamp_ms<?", (t1_ms, t2_ms)),
        ("warm_test", "m.timestamp_ms>=?", (t2_ms,)),
    ):
        connection.execute(f"CREATE TABLE {table_name} AS SELECT m.user_idx,m.item_idx,m.rating,m.timestamp_ms,m.source_row,? - p.prior_count AS candidate_count FROM mapped_p4 m JOIN user_time_prefix p USING(user_idx,timestamp_ms) WHERE {where_clause}", (item_count, *parameters))
        connection.execute(f"CREATE INDEX {table_name}_user ON {table_name}(user_idx)")

    artifacts = {
        "train_edges": export_query_gzip(connection, "SELECT user_idx,item_idx,rating,timestamp_ms,source_row FROM mapped_train ORDER BY source_row", output_dir / "baby_p4_train_edges.csv.gz", ["user_idx","item_idx","rating","timestamp_ms","source_row"]),
        "user_mapping": export_query_gzip(connection, "SELECT user_id,user_idx FROM user_map ORDER BY user_idx", output_dir / "baby_p4_user_mapping.csv.gz", ["user_id","user_idx"]),
        "item_mapping": export_query_gzip(connection, "SELECT item_id,item_idx FROM item_map ORDER BY item_idx", output_dir / "baby_p4_item_mapping.csv.gz", ["parent_asin","item_idx"]),
        "validation_targets": export_query_gzip(connection, "SELECT user_idx,item_idx,rating,timestamp_ms,source_row,candidate_count FROM warm_validation ORDER BY source_row", output_dir / "baby_p4_validation_targets.csv.gz", ["user_idx","item_idx","rating","timestamp_ms","source_row","candidate_count"]),
        "test_targets": export_query_gzip(connection, "SELECT user_idx,item_idx,rating,timestamp_ms,source_row,candidate_count FROM warm_test ORDER BY source_row", output_dir / "baby_p4_test_targets.csv.gz", ["user_idx","item_idx","rating","timestamp_ms","source_row","candidate_count"]),
    }

    user_degree = histogram_summary(connection.execute("SELECT degree,COUNT(*) FROM (SELECT COUNT(*) AS degree FROM mapped_train GROUP BY user_idx) GROUP BY degree ORDER BY degree"))
    item_degree = histogram_summary(connection.execute("SELECT degree,COUNT(*) FROM (SELECT COUNT(*) AS degree FROM mapped_train GROUP BY item_idx) GROUP BY degree ORDER BY degree"))
    components = connected_component_summary(connection, user_count, item_count)
    validation = partition_ledger(connection, "e.timestamp_ms>=? AND e.timestamp_ms<?", (t1_ms, t2_ms))
    test = partition_ledger(connection, "e.timestamp_ms>=?", (t2_ms,))
    dry_run = run_dry_feasibility(connection, item_count, dry_run_targets, evaluation_chunk_size)

    manifest = {
        "status": "G2_C_ARTIFACTS_BUILT; G2_D_BOUNDED_TRAVERSAL_EXECUTED; GATE_REVIEW_REQUIRED",
        "created_at_utc": utc_now(),
        "source": {"path": str(input_path), "bytes": input_path.stat().st_size, "sha256": sha256_file(input_path)},
        "policy": {
            "positive": "P4: rating >= 4 and rating <= 5",
            "rating_zero": "quarantined",
            "duplicate": "earliest timestamp then source-row order",
            "t1_ms": t1_ms,
            "t2_ms": t2_ms,
            "cutoff_status": "CANDIDATE_EXECUTED; FREEZE REQUIRES G2-C REVIEW",
            "timestamp_tie_rule": "strict half-open intervals; events exactly at t1 enter validation and exactly at t2 enter test",
            "training_filter": {"min_user_degree": min_user_degree, "min_item_degree": min_item_degree, "iterations": filter_ledger},
            "mapping": "lexicographic IDs from filtered training positives only",
            "evaluation_history": "all mapped P4 events with timestamp strictly earlier than target; same-timestamp events are not prior",
            "candidate_universe": "full frozen training-item universe minus prior mapped positives; target retained",
        },
        "input_counts": {"raw_rows": raw_rows, "invalid_rows": invalid_rows, "rating_zero_rows": anomaly_zero_rows, "p4_rows_before_pair_dedup": p4_rows, "p4_rows_after_pair_dedup": p4_deduplicated, "duplicate_p4_rows_removed": p4_rows - p4_deduplicated},
        "training_graph": {"edges": train_count, "users": user_count, "items": item_count, "density": fraction(train_count, user_count * item_count), "user_degree": user_degree, "item_degree": item_degree, "components": components},
        "partitions": {"validation": validation, "test": test},
        "artifacts": artifacts,
        "g2d_feasibility": dry_run,
        "claim_boundary": "No recommender was trained. No NDCG/Recall, method comparison, final profiling, or scalability claim is produced.",
    }
    connection.close()
    if cleanup_database and database_path.exists():
        database_path.unlink()
    return manifest


## Xây artifact và chạy bounded feasibility



In [ ]:
manifest_path = OUTPUT_DIR / 'baby_p4_g2c_manifest.json'
if manifest_path.is_file() and not FORCE_REBUILD_G2C:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['status'].startswith('G2_C_ARTIFACTS_BUILT'), manifest['status']
    print('REUSED_FROZEN_G2C_ARTIFACTS:', manifest_path)
else:
    manifest = build_g2c_artifacts(
        input_path=RAW_PATH,
        output_dir=OUTPUT_DIR,
        database_path=DATABASE_PATH,
        t1_ms=T1_MS,
        t2_ms=T2_MS,
        min_user_degree=MIN_TRAIN_USER_DEGREE,
        min_item_degree=MIN_TRAIN_ITEM_DEGREE,
        dry_run_targets=DRY_RUN_TARGETS,
        evaluation_chunk_size=EVALUATION_CHUNK_SIZE,
    )
    manifest["code_version"] = {
        "execution_artifact": "self-contained Colab notebook",
        "notebook_revision": NOTEBOOK_REVISION,
        "implementation_sha256": IMPLEMENTATION_SHA256,
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding='utf-8')
print(json.dumps({
    "status": manifest["status"],
    "training_graph": manifest["training_graph"],
    "partitions": manifest["partitions"],
    "g2d_feasibility": manifest["g2d_feasibility"],
    "manifest": str(manifest_path),
}, ensure_ascii=False, indent=2))


## Hoàn tất environment G2-D — chỉ chạy cell này sau khi mount Drive

Cell standalone này không rebuild G2-C graph đã chấp nhận. Cell verify năm artifact, capture exact Python/platform/CPU/RAM/Colab environment, replay bounded chunk-count traversal từ validation target đã lưu và bổ sung environment-completion record vào manifest hiện có.


In [ ]:
import csv, datetime as dt, gzip, hashlib, importlib.metadata, json, os, platform, resource, sqlite3, subprocess, sys, time
from pathlib import Path

if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

output_dir = Path('/content/drive/MyDrive/Phase2_Amazon_Audit/g2c_baby_p4')
manifest_path = output_dir / 'baby_p4_g2c_manifest.json'
assert manifest_path.is_file(), f'Missing completed manifest: {manifest_path}'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['status'].startswith('G2_C_ARTIFACTS_BUILT'), manifest['status']

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

artifact_readback = {}
for name, record in manifest['artifacts'].items():
    path = Path(record['path'])
    observed = {'exists': path.is_file()}
    if path.is_file():
        observed.update(bytes=path.stat().st_size, sha256=file_sha256(path))
    observed['bytes_match'] = observed.get('bytes') == record['bytes']
    observed['sha256_match'] = observed.get('sha256') == record['sha256']
    artifact_readback[name] = observed
assert all(v['exists'] and v['bytes_match'] and v['sha256_match'] for v in artifact_readback.values())

original = manifest['g2d_feasibility']
catalog_items = original['catalog_items']
chunk_size = original['chunk_size']
target_limit = original['targets_executed']
target_path = Path(manifest['artifacts']['validation_targets']['path'])
comparison_count = eligible_count = targets_replayed = 0
started = time.perf_counter()
with gzip.open(target_path, 'rt', encoding='utf-8', newline='') as handle:
    for row in csv.DictReader(handle):
        if targets_replayed >= target_limit:
            break
        candidate_count = int(row['candidate_count'])
        assert 1 <= candidate_count <= catalog_items
        for start in range(0, catalog_items, chunk_size):
            comparison_count += min(chunk_size, catalog_items - start)
        eligible_count += candidate_count
        targets_replayed += 1
wall_seconds = time.perf_counter() - started

def read_first_value(path, prefix):
    try:
        for line in Path(path).read_text(errors='replace').splitlines():
            if line.startswith(prefix):
                return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return None

memory_kib = read_first_value('/proc/meminfo', 'MemTotal')
memory_total_mib = float(memory_kib.split()[0]) / 1024 if memory_kib else None
try:
    gpu_query = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10, check=False,
    )
    gpu = gpu_query.stdout.strip() or None
except (OSError, subprocess.SubprocessError):
    gpu = None

package_versions = {}
for package in ('google-colab', 'ipykernel'):
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = None

completion = {
    'status': 'ENVIRONMENT_FINGERPRINT_AND_BOUNDED_REPLAY_EXECUTED',
    'completion_revision': 'g2d-env-completion-v1-2026-09-02',
    'captured_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    'environment': {
        'python_version': sys.version,
        'python_implementation': platform.python_implementation(),
        'python_executable': sys.executable,
        'sqlite_version': sqlite3.sqlite_version,
        'platform': platform.platform(),
        'system': platform.system(),
        'release': platform.release(),
        'machine': platform.machine(),
        'processor': platform.processor() or None,
        'cpu_model': read_first_value('/proc/cpuinfo', 'model name'),
        'logical_cpu_count': os.cpu_count(),
        'memory_total_mib': memory_total_mib,
        'gpu': gpu,
        'colab': {key: os.environ.get(key) for key in ('COLAB_RELEASE_TAG','COLAB_BACKEND_VERSION','COLAB_GPU','CUDA_VISIBLE_DEVICES')},
        'package_versions': package_versions,
    },
    'artifact_readback': artifact_readback,
    'bounded_replay': {
        'targets_replayed': targets_replayed,
        'catalog_items': catalog_items,
        'chunk_size': chunk_size,
        'catalog_comparisons': comparison_count,
        'eligible_candidates_counted': eligible_count,
        'wall_seconds': wall_seconds,
        'process_peak_rss_mb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
        'invariants': {
            'comparison_count_matches_original': comparison_count == original['catalog_comparisons'],
            'eligible_count_matches_original': eligible_count == original['eligible_candidates_counted'],
            'target_count_matches_original': targets_replayed == original['targets_executed'],
            'original_target_not_in_prior_history': original['invariants']['target_not_in_prior_history'],
        },
    },
    'claim_boundary': 'Environment/readback feasibility only; no model quality, comparative profiling, or scalability claim.',
}
assert all(completion['bounded_replay']['invariants'].values())
manifest['g2d_environment_completion'] = completion
manifest['status'] = 'G2_C_ARTIFACTS_BUILT; G2_D_ENVIRONMENT_COMPLETION_EXECUTED; GATE_REVIEW_REQUIRED'
temporary = manifest_path.with_suffix('.json.tmp')
temporary.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
temporary.replace(manifest_path)
print(json.dumps(completion, ensure_ascii=False, indent=2))


## Ranh giới diễn giải

Manifest và compressed artifact xác lập exact training graph, retained warm-start cohort, attrition ledger, candidate-count rule và bounded evaluator-path feasibility dưới configuration đã ghi. Dry run không tính NDCG/Recall và không phải final profiling có thể so sánh. G2-C/G2-D chỉ có thể pass sau khi read-back output, mọi count đối soát và cutoff/candidate rule được chấp nhận tường minh.
